In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# =========================
# Configuration
# =========================

ACCOUNT_SOURCE_PATH = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "accounts/accounts.csv"
)

SILVER_TABLE = "dbx_fintech_data_platform.silver.accounts"

QUARANTINE_TABLE = (
    "dbx_fintech_data_platform.silver.accounts_quarantine"
)

In [0]:
account_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ACCOUNT_SOURCE_PATH)
)

print("Source records:", account_df.count())

account_df.printSchema()

display(account_df)

In [0]:
account_clean_df = (
    account_df

    # String cleanup
    .withColumn("account_id", F.trim(F.col("account_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("account_type", F.upper(F.trim(F.col("account_type"))))
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("status", F.upper(F.trim(F.col("status"))))

    # Explicit timestamp standardization
    .withColumn("created_at", F.col("created_at").cast("timestamp"))
    .withColumn("updated_at", F.col("updated_at").cast("timestamp"))
)

display(account_clean_df)

In [0]:
valid_account_types = [
    "CHECKING",
    "CREDIT",
    "SAVINGS"
]

valid_currencies = [
    "EUR",
    "GBP",
    "INR",
    "USD"
]

valid_statuses = [
    "ACTIVE",
    "BLOCKED",
    "CLOSED"
]

In [0]:
valid_condition = (
    F.col("account_id").isNotNull() &
    (F.trim(F.col("account_id")) != "") &

    F.col("customer_id").isNotNull() &
    (F.trim(F.col("customer_id")) != "") &

    F.col("account_type").isNotNull() &
    F.col("account_type").isin(valid_account_types) &

    F.col("currency").isNotNull() &
    F.col("currency").isin(valid_currencies) &

    F.col("status").isNotNull() &
    F.col("status").isin(valid_statuses) &

    F.col("created_at").isNotNull() &
    F.col("updated_at").isNotNull() &

    (F.col("updated_at") >= F.col("created_at"))
)

In [0]:
valid_account_df = account_clean_df.filter(valid_condition)

invalid_account_df = account_clean_df.filter(~valid_condition)

print("Valid records:", valid_account_df.count())
print("Invalid records:", invalid_account_df.count())

In [0]:
quarantine_df = (
    invalid_account_df
    .withColumn(
        "dq_reason",
        F.when(
            F.col("account_id").isNull() |
            (F.trim(F.col("account_id")) == ""),
            "INVALID_ACCOUNT_ID"
        )
        .when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "INVALID_CUSTOMER_ID"
        )
        .when(
            F.col("account_type").isNull() |
            (~F.col("account_type").isin(valid_account_types)),
            "INVALID_ACCOUNT_TYPE"
        )
        .when(
            F.col("currency").isNull() |
            (~F.col("currency").isin(valid_currencies)),
            "INVALID_CURRENCY"
        )
        .when(
            F.col("status").isNull() |
            (~F.col("status").isin(valid_statuses)),
            "INVALID_STATUS"
        )
        .when(
            F.col("created_at").isNull(),
            "MISSING_CREATED_AT"
        )
        .when(
            F.col("updated_at").isNull(),
            "MISSING_UPDATED_AT"
        )
        .when(
            F.col("updated_at") < F.col("created_at"),
            "INVALID_TIMESTAMP_SEQUENCE"
        )
        .otherwise("UNKNOWN_DQ_ERROR")
    )
    .withColumn(
        "_quarantine_timestamp",
        F.current_timestamp()
    )
)

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS dbx_fintech_data_platform.silver
""")

In [0]:
(
    quarantine_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(QUARANTINE_TABLE)
)

In [0]:
customer_silver_df = spark.table(
    "dbx_fintech_data_platform.silver.customers"
)

invalid_customer_refs = (
    valid_account_df
    .select("customer_id")
    .distinct()
    .join(
        customer_silver_df
        .select("customer_id")
        .distinct(),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Invalid customer references:",
    invalid_customer_refs.count()
)

In [0]:
account_window = (
    Window
    .partitionBy("account_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("created_at").desc()
    )
)

ranked_account_df = (
    valid_account_df
    .withColumn(
        "_row_number",
        F.row_number().over(account_window)
    )
)

account_current_df = (
    ranked_account_df
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
account_final_df = account_current_df.select(
    "account_id",
    "customer_id",
    "account_type",
    "currency",
    "status",
    "created_at",
    "updated_at"
)

account_final_df.printSchema()

display(account_final_df)

In [0]:
(
    account_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
print(
    "Silver account records:",
    account_final_df.count()
)

In [0]:
print(
    "Unique accounts:",
    account_final_df
    .select("account_id")
    .distinct()
    .count()
)

In [0]:
duplicate_count = (
    account_final_df
    .groupBy("account_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate account IDs:",
    duplicate_count
)

In [0]:
final_invalid_refs = (
    account_final_df
    .select("customer_id")
    .distinct()
    .join(
        customer_silver_df
        .select("customer_id")
        .distinct(),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "Invalid customer references:",
    final_invalid_refs
)